# ASL Citizen Sharded Cache Builder
**One of 3 parallel copies -- run on a separate Kaggle account, each with a different `SHARD_ID`.**

This is the parallelized version of `asl_citizen_2000class_cache_builder.ipynb`: class
selection already happened once (in that notebook), so this copy only reads its assigned
1/3 shard of the still-uncached videos and caches those. **The only thing that should
differ between the 3 copies of this notebook is `CONFIG.SHARD_ID` in Section 2** (0, 1, or 2).

**Before running:**
1. Add the ASL Citizen dataset as input (same as the original cache-builder notebook).
2. Add the shard dataset (produced by `shard_split_snippet.py` in the original,
   already-running notebook) as input -- contains `nslt_shard_0/1/2.json` and
   `class_list_asl_citizen.txt`.
3. Set `CONFIG.SHARD_ID` below to 0, 1, or 2 -- a different value on each of the 3 accounts.

**After all 3 finish:** each account's `/kaggle/working/feature_cache_shard_{N}` becomes its
own output dataset. Merge all of them (plus the original partial cache) with the merge
snippet.

## 1. Setup & Installation

In [1]:
!pip install -q -U mediapipe opencv-python-headless einops


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 7.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [2]:
# Download the MediaPipe Tasks model bundles -- identical to v7 Section 1.
!wget -q -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
!wget -q -O pose_landmarker_lite.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task
!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

import os
for f in ["hand_landmarker.task", "pose_landmarker_lite.task", "face_landmarker.task"]:
    size_mb = os.path.getsize(f) / 1e6 if os.path.exists(f) else 0
    print(f"{f}: {'OK' if size_mb > 0.1 else 'MISSING/EMPTY'} ({size_mb:.1f} MB)")


hand_landmarker.task: OK (7.8 MB)
pose_landmarker_lite.task: OK (5.8 MB)
face_landmarker.task: OK (3.8 MB)


## Imports

In [3]:
import os
import re
import json
import random
import types
import glob
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import cv2

import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

print("MediaPipe:", mp.__version__)


MediaPipe: 0.10.35


## 2. Global Configuration

Same video/landmark parameters as the original cache-builder notebook (must match, since
these `.npz` files get merged with that notebook's output and fed into the same v8 training
notebook). The only new setting is `SHARD_ID` -- **change this to 0, 1, or 2** depending on
which account/copy this is.

In [4]:
class CONFIG:
    # ---- CHANGE THIS PER ACCOUNT: 0, 1, or 2 ----
    SHARD_ID = 0

    # ---- ASL Citizen input (adjust to your actual mounted path) ----
    ASL_CITIZEN_ROOT = Path("/kaggle/input/datasets/pranaysahaps/asl-citizen")
    ASL_CITIZEN_VIDEOS_DIR = ASL_CITIZEN_ROOT / "videos"

    # ---- Shard input (from shard_split_snippet.py, run in the original notebook) ----
    SHARD_DIR = Path("/kaggle/input/notebooks/tonimaislamlamia/asl-cache-building/shards")
    SHARD_FILE = SHARD_DIR / f"nslt_shard_{SHARD_ID}.json"
    CLASS_LIST_FILE = SHARD_DIR / "class_list_asl_citizen.txt"

    WORK_DIR = Path("/kaggle/working")
    CACHE_DIR = WORK_DIR / f"feature_cache_shard_{SHARD_ID}"

    # ---- MediaPipe Tasks API model bundles (downloaded in Section 1) ----
    HAND_MODEL_PATH = "hand_landmarker.task"
    POSE_MODEL_PATH = "pose_landmarker_lite.task"
    FACE_MODEL_PATH = "face_landmarker.task"

    SEED = 42

    # ---- Video / temporal sampling -- MUST match the original notebook exactly ----
    FRAME_COUNT = 16
    IMG_SIZE = 112
    NUM_APPEARANCE_STREAMS = 1
    SIGNER_BBOX_MARGIN = 0.25

    N_HAND_LANDMARKS = 21
    N_POSE_LANDMARKS = 33
    N_FACE_LANDMARKS = 10


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)


seed_everything(CONFIG.SEED)
CONFIG.CACHE_DIR.mkdir(parents=True, exist_ok=True)

n_cached_files = len(list(CONFIG.CACHE_DIR.glob("*.npz")))
print(f"Shard {CONFIG.SHARD_ID}: {n_cached_files} clips already cached in "
      f"{CONFIG.CACHE_DIR} (resumable -- safe to re-run if this session times out)")


Shard 0: 0 clips already cached in /kaggle/working/feature_cache_shard_0 (resumable -- safe to re-run if this session times out)


## 3. Load This Shard's Manifest

No class selection here -- that already happened once in the original cache-builder
notebook. This just loads the pre-built subset of videos assigned to this shard.

In [5]:
with open(CONFIG.SHARD_FILE, "r") as f:
    nslt_data = json.load(f)

class_list = {}
with open(CONFIG.CLASS_LIST_FILE, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        idx_str, gloss = line.split("\t", 1)
        class_list[int(idx_str)] = gloss

print(f"Shard {CONFIG.SHARD_ID}: {len(nslt_data):,} videos assigned, "
      f"{len(class_list)} total classes in vocabulary")

videos_dir = CONFIG.ASL_CITIZEN_VIDEOS_DIR


Shard 0: 21,183 videos assigned, 2000 total classes in vocabulary


## 5. MediaPipe Landmarkers (unchanged from v7)

`create_landmarkers`, the Tasks-API-to-legacy-shim (`to_legacy_style_results`), and
the handedness-by-screen-position fix are copied verbatim from v7 Section 5/6 --
this is the part of the pipeline that stays identical regardless of data source.

In [6]:
def create_landmarkers():
    hand_options = mp_vision.HandLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=CONFIG.HAND_MODEL_PATH),
        running_mode=mp_vision.RunningMode.VIDEO,
        num_hands=2,
        min_hand_detection_confidence=0.5,
    )
    pose_options = mp_vision.PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=CONFIG.POSE_MODEL_PATH),
        running_mode=mp_vision.RunningMode.VIDEO,
    )
    face_options = mp_vision.FaceLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=CONFIG.FACE_MODEL_PATH),
        running_mode=mp_vision.RunningMode.VIDEO,
        num_faces=1,
    )
    hand_landmarker = mp_vision.HandLandmarker.create_from_options(hand_options)
    pose_landmarker = mp_vision.PoseLandmarker.create_from_options(pose_options)
    face_landmarker = mp_vision.FaceLandmarker.create_from_options(face_options)
    return hand_landmarker, pose_landmarker, face_landmarker


class _LandmarkListWrapper:
    """Gives a Tasks-API landmark list a `.landmark` attribute, matching what
    the rest of the pipeline expects from the old solutions API."""
    def __init__(self, landmark_list):
        self.landmark = landmark_list


def to_legacy_style_results(hand_result, pose_result, face_result):
    left_hand_landmarks, right_hand_landmarks = None, None

    if hand_result.hand_landmarks:
        # Assign by screen x-position, not MediaPipe's handedness label --
        # the label is unreliable when hands are close together (v7 fix).
        hands_with_x = [(lm_list[0].x, lm_list) for lm_list in hand_result.hand_landmarks]
        hands_with_x.sort(key=lambda h: h[0])

        if len(hands_with_x) >= 1:
            left_hand_landmarks = _LandmarkListWrapper(hands_with_x[0][1])
        if len(hands_with_x) >= 2:
            right_hand_landmarks = _LandmarkListWrapper(hands_with_x[-1][1])

    pose_landmarks = None
    if pose_result.pose_landmarks:
        pose_landmarks = _LandmarkListWrapper(pose_result.pose_landmarks[0])

    face_landmarks = None
    if face_result.face_landmarks:
        face_landmarks = _LandmarkListWrapper(face_result.face_landmarks[0])

    return types.SimpleNamespace(
        left_hand_landmarks=left_hand_landmarks,
        right_hand_landmarks=right_hand_landmarks,
        pose_landmarks=pose_landmarks,
        face_landmarks=face_landmarks,
    )


## 6. Signer Bounding Box + Landmark Vector (unchanged from v7)

`get_signer_bbox`/`crop_signer` (the v7 audit's full-pose-box appearance input)
and `landmarks_to_vector` (the fixed-size flattened landmark feature) are copied
verbatim.

In [7]:
def get_hand_bbox(landmarks, frame_shape, margin=0.35):
    """Kept for parity with v7 -- not called by the appearance pipeline
    (superseded by get_signer_bbox below)."""
    if landmarks is None:
        return None
    h, w = frame_shape[:2]
    xs = [lm.x for lm in landmarks.landmark]
    ys = [lm.y for lm in landmarks.landmark]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    box_w, box_h = x_max - x_min, y_max - y_min
    x_min = max(0.0, x_min - box_w * margin)
    x_max = min(1.0, x_max + box_w * margin)
    y_min = max(0.0, y_min - box_h * margin)
    y_max = min(1.0, y_max + box_h * margin)
    return (int(x_min * w), int(y_min * h), int(x_max * w), int(y_max * h))


def get_signer_bbox(pose_landmarks, frame_shape, margin=CONFIG.SIGNER_BBOX_MARGIN):
    """Bounding box around the full pose skeleton (head to hips, shoulder to
    shoulder including extended arms). Falls back to the FULL FRAME when pose
    detection misses, so a missed detection never starves the appearance
    stream of any signal at all."""
    h, w = frame_shape[:2]
    if pose_landmarks is None:
        return (0, 0, w, h)
    xs = [lm.x for lm in pose_landmarks.landmark]
    ys = [lm.y for lm in pose_landmarks.landmark]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    box_w, box_h = x_max - x_min, y_max - y_min
    x_min = max(0.0, x_min - box_w * margin)
    x_max = min(1.0, x_max + box_w * margin)
    y_min = max(0.0, y_min - box_h * margin)
    y_max = min(1.0, y_max + box_h * margin)
    return (int(x_min * w), int(y_min * h), int(x_max * w), int(y_max * h))


def crop_signer(frame, bbox, out_size):
    x1, y1, x2, y2 = bbox
    if x2 <= x1 or y2 <= y1:
        return cv2.resize(frame, (out_size, out_size))
    patch = frame[y1:y2, x1:x2]
    return cv2.resize(patch, (out_size, out_size))


FACE_SUBSET_IDX = [70, 63, 105, 66, 107, 336, 296, 334, 293, 300]
POSE_SUBSET_IDX = [11, 12, 13, 14, 15, 16]  # shoulders, elbows, wrists


def landmarks_to_vector(results):
    """Flatten hands + pose subset + face subset into one normalized vector."""
    def _xy(lm_list, idx_list=None):
        if lm_list is None:
            n = len(idx_list) if idx_list else 0
            return np.zeros((n, 2), dtype=np.float32)
        pts = lm_list.landmark
        if idx_list is not None:
            pts = [pts[i] for i in idx_list]
        return np.array([[p.x, p.y] for p in pts], dtype=np.float32)

    left_hand = _xy(results.left_hand_landmarks, list(range(CONFIG.N_HAND_LANDMARKS)))
    right_hand = _xy(results.right_hand_landmarks, list(range(CONFIG.N_HAND_LANDMARKS)))
    pose = _xy(results.pose_landmarks, POSE_SUBSET_IDX)
    face = _xy(results.face_landmarks, FACE_SUBSET_IDX)

    if results.pose_landmarks is not None:
        l_sh = results.pose_landmarks.landmark[11]
        r_sh = results.pose_landmarks.landmark[12]
        center = np.array([(l_sh.x + r_sh.x) / 2, (l_sh.y + r_sh.y) / 2])
        scale = max(np.linalg.norm(
            np.array([l_sh.x, l_sh.y]) - np.array([r_sh.x, r_sh.y])), 1e-6)
    else:
        center, scale = np.array([0.5, 0.5]), 1.0

    all_pts = np.concatenate([left_hand, right_hand, pose, face], axis=0)
    all_pts = (all_pts - center) / scale
    return all_pts.flatten()


LANDMARK_VECTOR_DIM = (CONFIG.N_HAND_LANDMARKS * 2 + len(POSE_SUBSET_IDX)
                        + len(FACE_SUBSET_IDX)) * 2
print(f"Landmark vector dimension per frame: {LANDMARK_VECTOR_DIM}")


Landmark vector dimension per frame: 116


## 7. Feature Extraction + Cache Builder (speed fix vs. the first run)

Two changes from the version this cache was originally started with:

1. **The scan pass is removed.** It ran a full hand-detection pass over every raw
   frame to find which frames had a hand, then sampled from those -- built for
   WLASL's raw, untrimmed clips where the sign might occupy only part of a longer
   video. ASL Citizen's clips are already pre-trimmed to just the isolated sign, so
   that scan pass was pure overhead (roughly 2x the hand-landmarker calls per video
   for no real benefit). This version samples directly and uniformly over all raw
   frames instead -- expect roughly half the per-video time.
2. `build_feature_cache` resolves each video's path from
   `nslt_data[video_id]["video_file"]` (ASL Citizen's own filenames) rather than
   assuming a fixed `{video_id}.mp4` pattern.

**If you're resuming a run that already cached some videos with the old scan-based
version:** no action needed -- the `.npz` output format is identical either way, and
the skip-if-cached check means already-finished videos won't be reprocessed. The
first N videos in your cache were sampled with the scan pass, the rest with plain
uniform sampling; this is a minor, low-impact inconsistency (both are reasonable
frame-selection strategies for a pre-trimmed clip), not a correctness problem worth
re-caching over.

In [8]:
def load_frames(video_path, max_frames=None):
    cap = cv2.VideoCapture(str(video_path))
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if max_frames and len(frames) >= max_frames:
            break
    cap.release()
    return frames


def get_video_fps(video_path, default=30.0):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    return fps if fps and fps > 1 else default


def sample_frame_indices(total_frames, target_count):
    if total_frames <= 0:
        return [0] * target_count
    if total_frames >= target_count:
        return np.linspace(0, total_frames - 1, target_count).astype(int).tolist()
    idx = list(range(total_frames)) + [total_frames - 1] * (target_count - total_frames)
    return idx


def extract_dual_stream_features(video_path, hand_landmarker, pose_landmarker, face_landmarker,
                                  ts_counter, frame_count=CONFIG.FRAME_COUNT, img_size=CONFIG.IMG_SIZE):
    """v8 speed fix: no scan pass. The scan-then-resample step (finding which
    raw frames have a detected hand, then sampling from those) was built for
    WLASL's raw, untrimmed clips, where the sign might occupy only part of a
    longer video. ASL Citizen's clips are already pre-trimmed to just the
    isolated sign -- there's no "find the sign within a longer clip" problem
    to solve, so the scan pass was pure overhead here (roughly 2x the
    hand-landmarker calls per video for no real benefit). Straight uniform
    sampling over all raw frames instead."""
    frames = load_frames(video_path)
    if len(frames) == 0:
        return None, None

    indices = sample_frame_indices(len(frames), frame_count)

    appearance_seq = np.zeros((frame_count, CONFIG.NUM_APPEARANCE_STREAMS, img_size, img_size, 3),
                               dtype=np.uint8)
    landmark_seq = np.zeros((frame_count, LANDMARK_VECTOR_DIM), dtype=np.float32)

    for t, frame_idx in enumerate(indices):
        frame = frames[frame_idx]
        ts_counter[0] += 1
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)

        hand_result = hand_landmarker.detect_for_video(mp_image, ts_counter[0])
        pose_result = pose_landmarker.detect_for_video(mp_image, ts_counter[0])
        face_result = face_landmarker.detect_for_video(mp_image, ts_counter[0])
        results = to_legacy_style_results(hand_result, pose_result, face_result)

        signer_bbox = get_signer_bbox(results.pose_landmarks, frame.shape)
        appearance_seq[t, 0] = crop_signer(frame, signer_bbox, img_size)

        landmark_seq[t] = landmarks_to_vector(results)

    return appearance_seq, landmark_seq


def build_feature_cache(nslt_data, videos_dir, cache_dir, frame_count=CONFIG.FRAME_COUNT,
                         img_size=CONFIG.IMG_SIZE, limit=None):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    video_ids = list(nslt_data.keys())
    if limit is not None:
        video_ids = video_ids[:limit]

    hand_landmarker, pose_landmarker, face_landmarker = create_landmarkers()
    ts_counter = [0]

    n_cached, n_skipped, n_failed = 0, 0, 0
    try:
        for i, video_id in enumerate(video_ids):
            out_path = cache_dir / f"{video_id}.npz"
            if out_path.exists():
                n_skipped += 1
                continue
            # ASL Citizen filename adaptation: resolve from the manifest's
            # own "video_file" field rather than assuming a fixed extension.
            video_file = nslt_data[video_id]["video_file"]
            video_path = Path(videos_dir) / video_file
            if not video_path.exists():
                n_failed += 1
                continue
            appearance_seq, landmark_seq = extract_dual_stream_features(
                video_path, hand_landmarker, pose_landmarker, face_landmarker,
                ts_counter, frame_count, img_size)
            if appearance_seq is None:
                n_failed += 1
                continue
            np.savez_compressed(out_path, appearance=appearance_seq, landmarks=landmark_seq)
            n_cached += 1
            if (i + 1) % 200 == 0:
                print(f"  ...{i + 1}/{len(video_ids)} processed "
                      f"(cached={n_cached}, skipped={n_skipped}, failed={n_failed})")
    finally:
        hand_landmarker.close()
        pose_landmarker.close()
        face_landmarker.close()
    print(f"Done. Newly cached: {n_cached}  |  Already cached: {n_skipped}  |  Failed: {n_failed}")


## 8. Run the Cache Build (This Shard Only)

Resumable, same as the original notebook -- if this session times out, just re-run; the
skip-if-cached check picks up where it left off.

In [9]:
build_feature_cache(nslt_data, videos_dir, CONFIG.CACHE_DIR, limit=None)

n_final = len(list(CONFIG.CACHE_DIR.glob("*.npz")))
print(f"\nShard {CONFIG.SHARD_ID} cache now contains {n_final:,} / {len(nslt_data):,} "
      f"assigned clips ({100*n_final/len(nslt_data):.1f}%).")
print(f"\nNext: Save Version / create an output dataset from {CONFIG.CACHE_DIR}, "
      f"then merge it with the other 2 shards' outputs (and the original partial "
      f"cache) using the merge snippet.")


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1784651842.322142      86 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1784651842.343598      86 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1784651842.433747      91 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1784651842.470121      91 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1784651842.471991      95 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
W0000 00:00:1784651842.482253      98 inference_feedback_manager.cc:114] Feedback mana

  ...200/21183 processed (cached=200, skipped=0, failed=0)
  ...400/21183 processed (cached=400, skipped=0, failed=0)
  ...600/21183 processed (cached=600, skipped=0, failed=0)
  ...800/21183 processed (cached=800, skipped=0, failed=0)
  ...1000/21183 processed (cached=1000, skipped=0, failed=0)
  ...1200/21183 processed (cached=1200, skipped=0, failed=0)
  ...1400/21183 processed (cached=1400, skipped=0, failed=0)
  ...1600/21183 processed (cached=1600, skipped=0, failed=0)
  ...1800/21183 processed (cached=1800, skipped=0, failed=0)
  ...2000/21183 processed (cached=2000, skipped=0, failed=0)
  ...2200/21183 processed (cached=2200, skipped=0, failed=0)
  ...2400/21183 processed (cached=2400, skipped=0, failed=0)
  ...2600/21183 processed (cached=2600, skipped=0, failed=0)
  ...2800/21183 processed (cached=2800, skipped=0, failed=0)
  ...3000/21183 processed (cached=3000, skipped=0, failed=0)
  ...3200/21183 processed (cached=3200, skipped=0, failed=0)
  ...3400/21183 processed (cache